AI Bank Account Opening Fraud Detection

Model Training

This notebook trains and compares multiple Machine Learning and Deep Learning models for bank account opening fraud detection.

The notebook performs the following tasks:

- Load the processed dataset
- Train Machine Learning models
- Train Deep Learning models
- Compare model performance
- Save the trained models

Import Libraries

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping

import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Import Splited Data

In [19]:
X_train = joblib.load('/content/drive/MyDrive/PROYECT_4/data/processed/X_train.pkl')
X_test = joblib.load('/content/drive/MyDrive/PROYECT_4/data/processed/X_test.pkl')

X_train_scaled = joblib.load('/content/drive/MyDrive/PROYECT_4/data/processed/X_train_scaled.pkl')
X_test_scaled = joblib.load('/content/drive/MyDrive/PROYECT_4/data/processed/X_test_scaled.pkl')

y_train = joblib.load('/content/drive/MyDrive/PROYECT_4/data/processed/y_train.pkl')
y_test = joblib.load('/content/drive/MyDrive/PROYECT_4/data/processed/y_test.pkl')

In [20]:
import os

In [21]:
os.makedirs('/content/drive/MyDrive/PROYECT_4/models', exist_ok=True)

Create and Train Models

In [22]:
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'DecisionTree': Pipeline([
        ('model', DecisionTreeClassifier(
            max_depth=50,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'RandomForest': Pipeline([
        ('model', RandomForestClassifier(
            n_estimators=100,
            max_depth=30,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ]),

    'XGBoost': Pipeline([
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            random_state=42,
            n_jobs=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss'
        ))
    ])
}

In [23]:
for name, pipeline in models.items():
  print(f'Training: {name}')
  pipeline.fit(X_train, y_train)

  file_name = name.lower().replace(" ", "_")
  joblib.dump(pipeline, f'/content/drive/MyDrive/PROYECT_4/models/{file_name}.joblib')

  print(f'Saved model: {name}')

Training: LogisticRegression
Saved model: LogisticRegression
Training: DecisionTree
Saved model: DecisionTree
Training: RandomForest
Saved model: RandomForest
Training: XGBoost
Saved model: XGBoost


In [24]:
dnn_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.30),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.30),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

In [25]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [26]:
dnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

In [27]:
history = dnn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    batch_size=64,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9878 - loss: 0.0604 - precision: 0.0237 - recall: 0.0029 - val_accuracy: 0.9885 - val_loss: 0.0515 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9891 - loss: 0.0508 - precision: 0.5000 - recall: 0.0021 - val_accuracy: 0.9885 - val_loss: 0.0515 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9891 - loss: 0.0497 - precision: 1.0000 - recall: 0.0014 - val_accuracy: 0.9885 - val_loss: 0.0504 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9891 - loss: 0.0491 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9885 - val_loss: 0.0504 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9891 - loss: 0.0485 - precision: 0.5000

Update DNN Model with Class Weight

In [28]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

In [29]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.5055771479129143), np.int64(1): np.float64(45.32577903682719)}


In [30]:
history = dnn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    batch_size=64,
    epochs=50,
    callbacks=[early_stopping],
    verbose=1,
    class_weight=class_weights
)

Epoch 1/50
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7969 - loss: 0.4715 - precision: 0.0405 - recall: 0.7754 - val_accuracy: 0.7817 - val_loss: 0.4729 - val_precision: 0.0403 - val_recall: 0.7902
Epoch 2/50
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7868 - loss: 0.4343 - precision: 0.0410 - recall: 0.8269 - val_accuracy: 0.8336 - val_loss: 0.3458 - val_precision: 0.0496 - val_recall: 0.7439
Epoch 3/50
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7820 - loss: 0.4326 - precision: 0.0402 - recall: 0.8298 - val_accuracy: 0.7963 - val_loss: 0.4489 - val_precision: 0.0425 - val_recall: 0.7793
Epoch 4/50
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7851 - loss: 0.4193 - precision: 0.0409 - recall: 0.8312 - val_accuracy: 0.7797 - val_loss: 0.4411 - val_precision: 0.0406 - val_recall: 0.8038
Epoch 5/50
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7909 - loss: 0.4138 - precision: 0.0426 - recall: 0.8441 - val_accuracy: 0.8143 - va

Save Model

In [31]:
dnn_model.save('/content/drive/MyDrive/PROYECT_4/models/dnn_model.keras')

Multiple Machine Learning and Deep Learning models were successfully trained for bank account opening fraud detection.

The following tasks were completed:

- Loaded the processed training and testing datasets
- Built Scikit-learn Pipelines for the classical Machine Learning models
- Trained Logistic Regression, Decision Tree, Random Forest and XGBoost models
- Built and trained a Deep Neural Network using Keras Sequential
- Applied class weights to the Deep Neural Network to address the severe class imbalance
- Saved all trained models for evaluation in the next notebook

The trained models are now ready to be compared using fraud detection metrics such as Precision, Recall, F1-score, ROC-AUC and PR-AUC.